# India AQI — full pipeline walkthrough (for instructors)

This notebook **recreates the same execution order** as the Python project, with short explanations so you do not need to open every module under `src/`.

**Two paths in the codebase** (both exposed by `cli.py`):

| Path | CLI | What it does |
| --- | --- | --- |
| **EDA** | `python cli.py --stage <stage>` | National CSV → validate → clean → features → analysis blocks (profiling … full). Orchestrator: `src/pipelines/eda_pipeline.py` → `run_eda`. |
| **Prediction** | `python cli.py --predict --city …` | Chunked read **one city** → same validate/clean/features → multi-horizon Random Forest training. Orchestrator: `src/prediction/pipeline.py` → `run_prediction`. |

**How to use this notebook**

1. Run **Setup** first (adds the repo to `sys.path`, logging).
2. **EDA:** run cells in Part I in order. Set `EDA_STAGE` like the CLI (`profiling` … `full`). `full` runs every analysis step and writes all CSVs/plots (slow on the full dataset).
3. **Prediction:** open Part II; set `RUN_PREDICTION = True` when you want to train (optional; uses a separate data load path).

**Repository layout reminder:** `config/config.yaml` → `data.raw_path`; artifacts → `outputs/` (under the repo root, not the kernel’s cwd).


## Part 0 — Setup

Locates the directory that contains `cli.py` and `src/` (works if Jupyter’s cwd is the repo root **or** `notebooks/`). Initializes logging like the CLI.


In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "cli.py").is_file() and (d / "src").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find repo root (cli.py + src/). "
        "Start Jupyter from the repository root or from notebooks/."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.logger import setup_logger

setup_logger()
logging.getLogger().setLevel(logging.INFO)

print("Repository root:", REPO_ROOT)


---

## Part I — EDA pipeline (national dataset)

**Flow (same as `run_eda` in `src/pipelines/eda_pipeline.py`):**

1. **Ingestion** — `src/ingestion/loader.py`: read `config/config.yaml`, load CSV from `data.raw_path`, parse `Datetime`.
2. **Validation** — `src/preprocessing/validation.py`: required columns, range checks, duplicate row count.
3. **Cleaning** — `src/preprocessing/cleaning.py`: drop duplicate rows, clip/repair PM2.5/humidity/wind, sort by time.
4. **Feature engineering** — `src/preprocessing/feature_engineering.py`: calendar fields, lags, rolling stats, severity flags, weather/event flags.
5. **Analysis** (depends on stage): profiling → temporal → interactions → events + extremes → Tableau-style CSVs (`full`) → matplotlib figures (`visual` / `full`).

Set the stage below to mirror `cli.py --stage`.


In [ ]:
# Same choices as: python cli.py --stage <name>
EDA_STAGE = "profiling"  # profiling | temporal | interactions | events | visual | full


def _eda_runs_profiling(stage: str) -> bool:
    return stage in ("profiling", "full")


def _eda_runs_temporal(stage: str) -> bool:
    return stage in ("temporal", "full")


def _eda_runs_interactions(stage: str) -> bool:
    return stage in ("interactions", "full")


def _eda_runs_events(stage: str) -> bool:
    return stage in ("events", "full")


def _eda_runs_tableau(stage: str) -> bool:
    return stage == "full"


def _eda_runs_visual(stage: str) -> bool:
    return stage in ("visual", "full")


print("EDA_STAGE =", repr(EDA_STAGE))


### I.1 Ingestion (`src/ingestion/loader.py`)

`load_data()` reads YAML from `config/config.yaml`, resolves `data.raw_path` against the **repository root**, reads the CSV with pandas, and enforces core columns (`Datetime`, `PM2_5_ugm3`, `City`).


In [ ]:
from src.ingestion.loader import load_data

df = load_data()
print("Shape after load:", df.shape)
print("Columns (first 15):", list(df.columns[:15]), "...")


### I.2 Validation (`src/preprocessing/validation.py`)

- `validate_schema` — required columns exist.
- `validate_ranges` — logs counts of negative PM2.5, invalid humidity, negative wind (non-fatal).
- `validate_duplicates` — logs duplicate row count.


In [ ]:
from src.preprocessing.validation import (
    validate_schema,
    validate_ranges,
    validate_duplicates,
)

validate_schema(df)
validate_ranges(df)
validate_duplicates(df)
print("Validation steps completed.")


### I.3 Cleaning (`src/preprocessing/cleaning.py`)

Removes exact duplicates, drops negative PM2.5, clips humidity to 0–100, takes absolute wind, sorts by `City` / `Datetime` for consistent time series.


In [ ]:
from src.preprocessing.cleaning import clean_data

df = clean_data(df)
print("Shape after clean:", df.shape)


### I.4 Feature engineering (`src/preprocessing/feature_engineering.py`)

Adds time parts, PM2.5 lags (1h, 24h, 168h), rolling mean/std (24h), deviation from rolling baseline, Indian AQI-style categories, `is_severe`, weather interaction flags, and event-related fields used downstream.


In [ ]:
from src.preprocessing.feature_engineering import create_features

df = create_features(df)
print("Shape after features:", df.shape)
feat_cols = [c for c in df.columns if c.startswith("pm25") or c in ("is_severe", "year", "month", "hour")]
print("Sample engineered columns:", feat_cols[:20])


### I.5a Profiling (`src/analysis/profiling.py`)

Distribution tables, tail risk, missingness, per-city summaries. Writes `outputs/tables/*_profiling.csv` (paths are repo-root–absolute in code).


In [ ]:
from src.analysis.profiling import run_profiling

if _eda_runs_profiling(EDA_STAGE):
    run_profiling(df)
else:
    print("Skip profiling (EDA_STAGE does not include profiling).")


### I.5b Temporal analysis (`src/analysis/temporal.py`)

Daily/monthly aggregates, rolling trends, diurnal/seasonal patterns, lag correlation (Delhi slice), volatility. Outputs `*_temporal.csv`.


In [ ]:
from src.analysis.temporal import temporal_analysis

if _eda_runs_temporal(EDA_STAGE):
    temporal_analysis(df)
else:
    print("Skip temporal.")


### I.5c Interactions (`src/analysis/interactions.py`)

Wind × season, humidity, events × season, severe probability, city-level interaction tables → `*_interaction.csv`.


In [ ]:
from src.analysis.interactions import interaction_analysis

if _eda_runs_interactions(EDA_STAGE):
    interaction_analysis(df)
else:
    print("Skip interactions.")


### I.5d Events (`src/analysis/events.py`)

Festival and crop-burning style contrasts vs baseline, amplification, severity → `*_events.csv`.


In [ ]:
from src.analysis.events import event_analysis

if _eda_runs_events(EDA_STAGE):
    event_analysis(df)
else:
    print("Skip events.")


### I.5e Extremes (`src/analysis/extremes.py`)

Severe pollution conditions, distributions, probabilities → `*_extremes.csv`.


In [ ]:
from src.analysis.extremes import extreme_analysis

if _eda_runs_events(EDA_STAGE):
    extreme_analysis(df)
else:
    print("Skip extremes (same gate as events in run_eda).")


### I.5f Tableau-oriented datasets (`src/analysis/synthesis.py`)

Runs only for **`EDA_STAGE == "full"`**. Builds overview/temporal/city/interaction/events/extremes dashboard tables under `outputs/tables/Tableau/*_dashboard.csv`.


In [ ]:
from src.analysis.synthesis import build_all_datasets

if _eda_runs_tableau(EDA_STAGE):
    build_all_datasets(df)
else:
    print("Skip Tableau exports (only for stage 'full').")


### I.5g Visualizations (`src/analysis/visualization.py`)

Matplotlib/Seaborn PNGs under `outputs/plots/` for `visual` or `full`.


In [ ]:
if _eda_runs_visual(EDA_STAGE):
    from src.analysis.visualization import run_visualizations

    run_visualizations(df)
else:
    print("Skip matplotlib figures (use visual or full).")


### I.6 EDA wrap-up

The engineered `DataFrame` in memory is the same object passed through all steps. Below: optional peek at rows/columns.


In [ ]:
print("Final in-memory shape:", df.shape)
df[["Datetime", "City", "PM2_5_ugm3"]].head()


---

## Part II — Prediction pipeline (single city)

**Separate from EDA.** Implemented in `src/prediction/pipeline.py` → `run_prediction`:

1. `load_city_data_for_forecast(city)` — chunked read of the **national** CSV, keeps only rows for `city` and columns needed for modeling (`src/prediction/loader.py`).
2. Same **validate → clean → create_features** as above.
3. `run_forecast` in `src/analysis/forecasting.py` — adds horizon targets, time-based train/test split, fits **RandomForestRegressor** per horizon, saves `joblib` bundles and `manifest.json` under `outputs/prediction/models/<City>/<run_id>/`, writes `outputs/prediction/last_run.json`.

The CLI prints Rich tables; here we call the same function (set flag to run).


In [ ]:
# Set True to train (reads raw data again; default horizons are 1,6,12,24,48 hours)
RUN_PREDICTION = False
PREDICT_CITY = "Delhi"
PREDICT_HORIZONS = (1, 6, 12, 24, 48)  # or e.g. (1,) for a quick demo
PREDICT_TEST_FRACTION = 0.2

if RUN_PREDICTION:
    from src.prediction.pipeline import run_prediction

    summary = run_prediction(
        PREDICT_CITY,
        model_id="rf_multih_rf",
        horizons=PREDICT_HORIZONS,
        test_fraction=PREDICT_TEST_FRACTION,
    )
    print("Keys in summary:", sorted(summary.keys()))
else:
    print("Prediction skipped (set RUN_PREDICTION = True to run).")


---

## Source map (quick reference)

| Area | Main modules |
| --- | --- |
| CLI | `cli.py` |
| EDA orchestration | `src/pipelines/eda_pipeline.py` |
| Load / config | `src/ingestion/loader.py` |
| Preprocess | `src/preprocessing/validation.py`, `cleaning.py`, `feature_engineering.py` |
| EDA analysis | `src/analysis/profiling.py`, `temporal.py`, `interactions.py`, `events.py`, `extremes.py`, `synthesis.py`, `visualization.py` |
| Forecast math | `src/analysis/forecasting.py` |
| Prediction orchestration | `src/prediction/pipeline.py`, `src/prediction/loader.py` |
| Paths / logging | `src/utils/helpers.py` (`project_root`, `outputs_path`), `src/utils/logger.py` |
